# Kagle inclass https://www.kaggle.com/c/simplesentiment/overview

In [1]:
from itertools import product
from collections import Counter
import os
from pathlib import Path
import re

# import dill as pkl
import numpy as np
import nltk
from nltk.corpus import stopwords
import nltk.stem as st
import pandas as pd
#import plotly.graph_objects as go
import plotly.figure_factory as ff
# from tqdm import tqdm

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression#, SGDClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

In [3]:
from utilities import seedEverything

In [4]:
rand_seed = 141592
seedEverything(rand_seed)

## Задаю переменные

In [5]:
PATH_DATA = os.path.join(Path.cwd(), 'data')
PATH_SUBM = os.path.join(Path.cwd(), 'submissions')
PATH_MODL = os.path.join(Path.cwd(), 'models')

In [19]:
nltk.download('omw-1.4')  # для лемматизации 

lemm_eng = st.WordNetLemmatizer()  # влияет только на английские слова
stemm_eng = st.RSLPStemmer()  # st.ISRIStemmer()
stemm_ru = st.snowball.SnowballStemmer('russian')

[nltk_data] Downloading package omw-1.4 to /home/v010ch/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Загрузка, очистка и преобразование данных

In [7]:
df = pd.read_csv(os.path.join(PATH_DATA, 'ru_train.csv'), )
if 'Unnamed: 0' in df.columns:
    df.drop(['Unnamed: 0'], axis = 1, inplace = True)
df['target'] = df.rating.apply(lambda x: 0 if int(x) >= 4 else 1)

df.shape, df['target'].value_counts(normalize=True)

((4850, 6),
 target
 0    0.548454
 1    0.451546
 Name: proportion, dtype: float64)

In [8]:
stop_words = stopwords.words('russian')

clean_text = lambda x: re.sub(r"\s+", ' ', 
                              re.sub(r"[\d+]", '',
                                     re.sub(r"[^\w\s]", ' ', x.lower()).strip()
                                    )
                             )
# отзывы содержат наименования телефонов, очищаю от них и рускоязычных 
# вариантов названий, т.к. они могут привести к переобучению.
phones = r'\b(samsung|galaxy|xiaomi|iphon|redmi|note|honor|huawei|apple|' +\
          'nokia|meizu|google|самсунг|айфон|mi|lenovo|lg|redme|asus|vivo|' +\
          'zte|helio|mediatek|oppo|htc|pixel|xperia|fly|realme|zenfone|' +\
          'alcatel|blade|philips|touch|lumia|oneplus|motorola|inoi|red|neo|' +\
          'moto|panasonic|band|honnor|bbk|vertex|lafleur|xiomi|редми|хонор|' +\
          'ноки|хуаве|мейзу|асус|галакси|иной|гэлакси|хонор)(|[a-zа-я]+)\b'
clean_names = lambda x: re.sub(phones, '', x)
clean_stopwords = lambda x: ' '.join([word for word in x.split() if word not in stop_words])

# приведение к начальным формам
lem_eng_text = lambda x: ' '.join([lemm_eng.lemmatize(el) for el in x.split()])
stem_eng_text = lambda x: ' '.join([stemm_eng.stem(el) for el in x.split()])
stem_ru_text = lambda x: ' '.join([stemm_ru.stem(el) for el in x.split()])

In [9]:
%%time
df['text_cl'] = df.review.map(clean_text)
df['text_cl'] = df.text_cl.map(clean_names)
df['text_cl'] = df.text_cl.map(clean_stopwords)
df['text_cl'] = df.text_cl.map(lem_eng_text)
df['text_cl'] = df.text_cl.map(stem_eng_text)
df['text_cl'] = df.text_cl.map(stem_ru_text)

df = df.sample(frac=1).reset_index(drop=True)
df.sample(3)[['review', 'rating', 'link', 'target', 'text_cl']]

CPU times: user 53.6 s, sys: 177 ms, total: 53.8 s
Wall time: 53.8 s


,phone,review,rating,link,review_length,target,text_cl
3389,Смартфон Apple Iphone 5C,"Не зря Apple считают одной из лучших фирм, пот...",5.0,https://irecommend.ru/content/moya-prelest-282,464,0,зря appl счита одн лучш фирм техник действител...
1668,Nokia 5130 XpressMusic,Купила это телефон примерно год назад за 5000 ...,1.0,https://irecommend.ru/content/polnyi-uzhas-foto,69,1,куп эт телефон примерн год назад рубл снача не...
1883,Смартфон Samsung Galaxy J3 SM-J320F 2016,Вся моя семья фанаты сотовых телефонов фирмы s...,5.0,https://irecommend.ru/content/moya-lyubov-k-sm...,155,0,вся сем фанат сотов телефон фирм samsung одн в...


## Создание и сохранение модели и токенайзера для инференса и демонстрации на flask

Очищенные отзывы векторизую через tf-idf.   
Полученные векторы в LogReg для подбора параметров.

In [10]:
grid_pipe = Pipeline([('vector', TfidfVectorizer(analyzer = 'char_wb')),
                     ('model', LogisticRegression(solver = 'liblinear',))
                     ])

params_pipe = {'vector__max_features': [768, 1024],       # 256, 512, 768, 
               'vector__ngram_range': [(2, 3), (3, 4) ],  # (2, 4)
               'vector__min_df': [0.2, 0.3],              # 0.5
               'vector__max_df': [0.75, 0.8],             # 0.7, 1.0
               'model__penalty': ['l2'],                  # 'l1'
               'model__class_weight': ['balanced'],   #{0: 0.5, 1: 0.5}, {0: 0.5894081632653061, 1: 0.41059183673469385},
               'model__max_iter': [ 20, 100],             # 10, 50],
               'model__C': [2, 3, 4]
               }

In [11]:
%%time
clf = GridSearchCV(grid_pipe, params_pipe,
                   n_jobs=-1, cv=5,
                   scoring = ['roc_auc', 'accuracy'],
                   refit='roc_auc',
                   verbose=0,
                  )
clf.fit(df.text_cl, df.target)

CPU times: user 7.91 s, sys: 1.53 s, total: 9.43 s
Wall time: 9min 29s


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('vector',
                                        TfidfVectorizer(analyzer='char_wb')),
                                       ('model',
                                        LogisticRegression(solver='liblinear'))]),
             n_jobs=-1,
             param_grid={'model__C': [2, 3, 4],
                         'model__class_weight': ['balanced'],
                         'model__max_iter': [20, 100], 'model__penalty': ['l2'],
                         'vector__max_df': [0.75, 0.8],
                         'vector__max_features': [768, 1024],
                         'vector__min_df': [0.2, 0.3],
                         'vector__ngram_range': [(2, 3), (3, 4)]},
             refit='roc_auc', scoring=['roc_auc', 'accuracy'])

In [12]:
clf.best_score_, clf.best_params_

(np.float64(0.9167361897895423),
 {'model__C': 4,
  'model__class_weight': 'balanced',
  'model__max_iter': 20,
  'model__penalty': 'l2',
  'vector__max_df': 0.75,
  'vector__max_features': 1024,
  'vector__min_df': 0.2,
  'vector__ngram_range': (2, 3)})

 Сохраняю результаты на всех возможных комбинациях заданных параметров   
 для дальнейшего анализа

In [13]:
res = pd.DataFrame(clf.cv_results_['params'])
res['mean_test_roc_auc'] = clf.cv_results_['mean_test_roc_auc']
res['mean_test_accuracy'] = clf.cv_results_['mean_test_accuracy']
res['mean_sum'] = (clf.cv_results_['mean_test_accuracy'] + clf.cv_results_['mean_test_roc_auc'])/2

res.to_csv(os.path.join(PATH_DATA, 'grid1_res.csv'), index=False)

## Обучаю модель на лучших параметрах и полных данных

In [14]:
%%time
vectorizer = TfidfVectorizer(analyzer = 'char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                             max_features=clf.best_params_['vector__max_features'],
                            )
vectorizer.fit(df.text_cl)
train = vectorizer.transform(df.text_cl)

model = LogisticRegression(penalty=clf.best_params_['model__penalty'],
                           solver='liblinear',
                           C=clf.best_params_['model__C'],
                           class_weight=clf.best_params_['model__class_weight'],
                           max_iter=clf.best_params_['model__max_iter'],
                          )
model.fit(train, df.target)

CPU times: user 7.96 s, sys: 60.8 ms, total: 8.02 s
Wall time: 8.03 s


LogisticRegression(C=4, class_weight='balanced', max_iter=20,
                   solver='liblinear')

## Сохраняю

С учетом предполагаемого применения, будет необходимо отслеживать версии    
скриптов очистки, приведения к начальной форме и модели. Для облегчения   
этолй задачи объеденим все в класс. При изменении версий данный класс не   
будет занимать много места, об этом не беспокоюсь.    

## Посмотрим на результат обучения на полном датасете

In [20]:
pred_train_tfidf = model.predict(train)
print( f'roc_auc_score: {roc_auc_score(df.target, pred_train_tfidf)}\n'\
       f'accuracy_score: {accuracy_score(df.target, pred_train_tfidf)}')

roc_auc_score: 0.8851735503141416
accuracy_score: 0.8863917525773196


In [21]:
cm = confusion_matrix(df.target, pred_train_tfidf)
cm

array([[2388,  272],
       [ 279, 1911]])

## Ошибки в предсказаниях обзоров

In [22]:
df['pred_train_tfidf'] = pred_train_tfidf
error_reviews = df[df['target'] != df['pred_train_tfidf']]

In [23]:
tmp = error_reviews.sample()
print(f'True class {tmp.target.item()},\nPredicted class {tmp.pred_train_tfidf.item()}\n')
print(tmp.review.item())

True class 1,
Predicted class 0

Год назад из России тетя прислала мне смартфон LA'Fleur GT-5380....
ВАУ!!!
on samyi
Круто точно такой же как по ТВ показывали вау я крутая' подумала я))))) Сначала он мне нравился, он казался очень красивым и сенсорные клавиатуры мне нравились. Но.... Потом я стала понимать что с ним скучно, да это круто ютуб,интернет но почему play marketa нету????????!!!!!????Только samsungapps..фу блина это разрдрожала,я не могла смотреть видео в HD качестве. Не могла в браузере запустить видео, через год только получилось.
Не могла скачать нужные приложения whatsapp или skype, ну и приложения для изучения языков для этого надо было купить их за 0.99$ че за фигня на андроиде бесплатно а на баде нет значит!
?В соц сетях могла только сидеть в одноклассниках, в фейсе,в як,в яху,в чатоне и вк! А самое обидное было то что задавали тупой вопрос который так популярен "А это китайский аппарат? " какой на фиг китайский они че телевизор не смотрят! Сколько показывали.
Зарядки 